# TPCRP Active Learning on CIFAR-10
**Paper:** *Active Learning on a Budget: Opposite Strategies Suit High and Low Budgets*  
Hacohen, Dekel & Weinshall — ICML 2022

Implementation from scratch following the paper's exact specifications (Appendix F).

## Cell 1 — Imports, Constants, and Seed Utilities

In [ ]:
import os, random, pickle, csv, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Directories ───────────────────────────────────────────────────────────────
DATA_DIR   = "./data"
OUTPUT_DIR = "./output"
os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── SimCLR hyperparameters (paper Appendix F.1) ───────────────────────────────
SIMCLR_EPOCHS     = 500
SIMCLR_BATCH_SIZE = 512
SIMCLR_LR         = 0.4
SIMCLR_MOMENTUM   = 0.9
SIMCLR_WD         = 1e-4
SIMCLR_TEMP       = 0.5
PROJ_DIM          = 128   # projection head output dimension
EMBED_DIM         = 512   # ResNet-18 penultimate layer dimension

# ── Active-learning hyperparameters (paper Section 3.2 / Appendix F.1) ────────
AL_BUDGET      = 10   # B: labels queried per round
AL_ITERATIONS  = 5    # number of AL rounds
KNN_K          = 20   # nearest neighbours for typicality (paper footnote 1)
MAX_CLUSTERS   = 500  # cap on K-means clusters (paper Appendix F.1)
N_REPETITIONS  = 10   # repetitions for mean ± SE (paper Section 4.2)
N_TRAIN        = 50_000
N_TEST         = 10_000

# ── Classifier hyperparameters ────────────────────────────────────────────────
CLF_EPOCHS   = 100
CLF_LR       = 0.025
CLF_MOMENTUM = 0.9
CLF_WD       = 5e-4

# ── CIFAR-10 normalisation constants ─────────────────────────────────────────
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)


def set_seed(seed: int) -> None:
    """Set all relevant random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


set_seed(42)
print("Constants and seed utilities initialised.")


## Cell 2 — SimCLR Model Architecture and NT-Xent Loss

- **Backbone**: ResNet-18 (no pretrained weights)  
- **Projection head**: 2-layer MLP, hidden dim = 512, output dim = 128  
- **Loss**: NT-Xent (normalised temperature-scaled cross-entropy) with τ = 0.5

In [ ]:
class SimCLRProjectionHead(nn.Module):
    """2-layer MLP: EMBED_DIM → EMBED_DIM (ReLU) → PROJ_DIM."""

    def __init__(self, in_dim: int = EMBED_DIM, hidden_dim: int = EMBED_DIM,
                 out_dim: int = PROJ_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class SimCLR(nn.Module):
    """
    ResNet-18 backbone + SimCLR projection head.

    forward() returns:
      h : (B, 512) penultimate-layer features  (L2-normalised at extraction time)
      z : (B, 128) projection-head outputs     (used only during contrastive training)
    """

    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=None)
        # Drop the final linear classifier; keep everything up to and including avgpool
        self.encoder   = nn.Sequential(*list(backbone.children())[:-1])  # → (B, 512, 1, 1)
        self.projector = SimCLRProjectionHead()

    def forward(self, x: torch.Tensor):
        h = self.encoder(x).flatten(1)   # (B, 512)
        z = self.projector(h)             # (B, 128)
        return h, z


def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor,
                 temperature: float = SIMCLR_TEMP) -> torch.Tensor:
    """
    NT-Xent contrastive loss (Chen et al., 2020).

    Given N paired views (z1_i, z2_i), constructs a 2N * 2N similarity matrix
    and treats each pair as a positive while all 2(N-1) others are negatives.

    Args:
        z1, z2      : (N, D) raw (un-normalised) projection vectors
        temperature : τ scaling factor

    Returns:
        scalar loss
    """
    N = z1.size(0)
    # L2-normalise both views
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    z  = torch.cat([z1, z2], dim=0)   # (2N, D)

    # Full pairwise cosine-similarity matrix, scaled by temperature
    sim = torch.mm(z, z.T) / temperature   # (2N, 2N)

    # Mask out diagonal (self-similarity = trivial positive)
    diag_mask = torch.eye(2 * N, dtype=torch.bool, device=z.device)
    sim.masked_fill_(diag_mask, float("-inf"))

    # Positive-pair labels:
    #   row  i   → positive at i+N
    #   row i+N  → positive at i
    labels = torch.cat([
        torch.arange(N, 2 * N, device=z.device),
        torch.arange(0, N,     device=z.device),
    ])  # (2N,)

    return F.cross_entropy(sim, labels)


print("SimCLR architecture and NT-Xent loss defined.")


## Cell 3 — SimCLR Data Augmentation Pipeline

Augmentations from paper Appendix F.1:  
random resized crop (scale 0.2–1.0), horizontal flip, colour jitter (brightness/contrast/saturation = 0.4, hue = 0.1), random greyscale (p = 0.2), Gaussian blur (applied with p = 0.5, kernel 3 * 3 for 32 * 32 images).

In [ ]:
class TwoViewTransform:
    """Wraps a transform and applies it independently twice to produce two augmented views."""

    def __init__(self, transform):
        self.transform = transform

    def __call__(self, x):
        return self.transform(x), self.transform(x)


def get_simclr_transform() -> T.Compose:
    """
    SimCLR augmentation pipeline for CIFAR-10 (32 * 32 px).
    Matches paper Appendix F.1 exactly.
    """
    colour_jitter = T.ColorJitter(
        brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1
    )
    return T.Compose([
        T.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
        T.RandomHorizontalFlip(),
        T.RandomApply([colour_jitter], p=0.8),
        T.RandomGrayscale(p=0.2),
        # Gaussian blur: kernel_size = 0.1 * image_size ≈ 3 for CIFAR-10
        T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.5),
        T.ToTensor(),
        T.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])


def get_eval_transform() -> T.Compose:
    """Plain deterministic transform for embedding extraction."""
    return T.Compose([
        T.ToTensor(),
        T.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])


print("SimCLR augmentation pipelines defined.")


## Cell 4 — SimCLR Training (Step 1)

Trains SimCLR on all 50,000 unlabelled CIFAR-10 training images for 500 epochs. If a checkpoint already exists on disk the training step is skipped.

In [ ]:
def train_simclr() -> tuple[np.ndarray, np.ndarray]:
    """
    Train SimCLR on the full (unlabelled) CIFAR-10 training set.

    Saves:
      output/simclr_model.pt   — trained model state dict
      output/embeddings.npy    — (50000, 512) L2-normalised penultimate embeddings
      output/train_labels.npy  — (50000,) ground-truth labels (oracle, used only for eval)

    Returns:
      embeddings   : (N_TRAIN, EMBED_DIM) float32 array
      train_labels : (N_TRAIN,) int array
    """
    ckpt_path       = os.path.join(OUTPUT_DIR, "simclr_model.pt")
    embeddings_path = os.path.join(OUTPUT_DIR, "embeddings.npy")
    labels_path     = os.path.join(OUTPUT_DIR, "train_labels.npy")

    # ── Load from disk if available ───────────────────────────────────────────
    if os.path.exists(embeddings_path) and os.path.exists(labels_path):
        print("[SimCLR] Checkpoint found — loading embeddings from disk.")
        return np.load(embeddings_path), np.load(labels_path)

    set_seed(42)

    # ── Dataset with two-view transform ───────────────────────────────────────
    train_ds = torchvision.datasets.CIFAR10(
        root=DATA_DIR, train=True, download=True,
        transform=TwoViewTransform(get_simclr_transform()),
    )
    loader = DataLoader(
        train_ds,
        batch_size=SIMCLR_BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        drop_last=True,     # keeps all batches the same size for NT-Xent
    )

    # ── Model, optimiser, scheduler ───────────────────────────────────────────
    model     = SimCLR().to(DEVICE)
    optimiser = optim.SGD(
        model.parameters(),
        lr=SIMCLR_LR, momentum=SIMCLR_MOMENTUM, weight_decay=SIMCLR_WD,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=SIMCLR_EPOCHS)

    # ── Training loop ─────────────────────────────────────────────────────────
    model.train()
    print(f"[SimCLR] Starting training for {SIMCLR_EPOCHS} epochs …")
    for epoch in range(1, SIMCLR_EPOCHS + 1):
        epoch_loss = 0.0
        n_batches  = 0
        for (x1, x2), _ in loader:
            x1, x2 = x1.to(DEVICE), x2.to(DEVICE)
            optimiser.zero_grad()
            _, z1 = model(x1)
            _, z2 = model(x2)
            loss  = nt_xent_loss(z1, z2, SIMCLR_TEMP)
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item()
            n_batches  += 1
        scheduler.step()

        if epoch % 50 == 0 or epoch == 1:
            avg = epoch_loss / n_batches
            lr  = scheduler.get_last_lr()[0]
            print(f"  Epoch [{epoch:3d}/{SIMCLR_EPOCHS}]  avg_loss={avg:.4f}  lr={lr:.6f}")

    torch.save(model.state_dict(), ckpt_path)
    print(f"[SimCLR] Model saved → {ckpt_path}")

    # ── Extract and save embeddings ───────────────────────────────────────────
    embeddings, train_labels = _extract_embeddings(model)
    np.save(embeddings_path, embeddings)
    np.save(labels_path,     train_labels)
    print(f"[SimCLR] Embeddings saved → {embeddings_path}  shape={embeddings.shape}")

    return embeddings, train_labels


def _extract_embeddings(model: SimCLR) -> tuple[np.ndarray, np.ndarray]:
    """
    Extract L2-normalised 512-dim penultimate-layer representations from a
    trained SimCLR model for all 50,000 CIFAR-10 training images.
    """
    ds = torchvision.datasets.CIFAR10(
        root=DATA_DIR, train=True, download=True,
        transform=get_eval_transform(),
    )
    loader = DataLoader(ds, batch_size=512, shuffle=False,
                        num_workers=4, pin_memory=True)
    model.eval()
    all_h, all_y = [], []
    with torch.no_grad():
        for x, y in loader:
            x    = x.to(DEVICE)
            h, _ = model(x)
            h    = F.normalize(h, dim=1)   # L2-normalise (paper Appendix F.1)
            all_h.append(h.cpu().numpy())
            all_y.append(y.numpy())
    return np.concatenate(all_h), np.concatenate(all_y)


print("SimCLR training function defined.")


## Cell 5 — Run SimCLR Training and Extract Embeddings

In [ ]:
print("=" * 60)
print("Step 1 — SimCLR Representation Learning")
print("=" * 60)
embeddings, train_labels = train_simclr()
print(f"Embeddings : {embeddings.shape}  dtype={embeddings.dtype}")
print(f"Labels     : {train_labels.shape}  classes={np.unique(train_labels)}")
print(f"L2 norms   : min={np.linalg.norm(embeddings, axis=1).min():.4f}  "
      f"max={np.linalg.norm(embeddings, axis=1).max():.4f}")
print(f"Class distribution: {np.bincount(train_labels)}")


## Cell 6 — TPCRP Active Learning Selection (Algorithm 1)

Implements Algorithm 1 from the paper with the TPCRP variant (SimCLR embeddings + K-means clustering):

1. Cluster all data into |L| + B clusters (K-means)  
2. Identify uncovered clusters (no labeled point inside)  
3. From the B *largest* uncovered clusters, select the point with highest Typicality  

**Typicality(x)** = 1 / mean Euclidean distance to K=20 nearest neighbours (Equation 4 of the paper).

In [ ]:
def compute_typicality_global(
    embeddings: np.ndarray,
    query_indices: list[int],
    k: int = KNN_K,
) -> np.ndarray:
    """
    Compute Typicality (Eq. 4) for a list of query indices.
    KNN is fit on *all* N embeddings — density is measured in the full space.

    Args:
        embeddings    : (N, D) array of all embeddings
        query_indices : list of integer indices to compute typicality for
        k             : number of nearest neighbours (K = 20 per paper)

    Returns:
        typicality : (len(query_indices),) array  — higher = more typical
    """
    k_eff = min(k, len(embeddings) - 1)
    nbrs = NearestNeighbors(
        n_neighbors=k_eff + 1,   # +1 because the point itself is its own NN
        metric="euclidean",
        algorithm="auto",
        n_jobs=-1,
    )
    nbrs.fit(embeddings)
    query   = embeddings[np.array(query_indices)]
    dists, _ = nbrs.kneighbors(query)      # (|query|, k_eff+1)
    mean_dists = dists[:, 1:].mean(axis=1) # exclude self (col 0), shape (|query|,)
    return 1.0 / (mean_dists + 1e-10)


def tpcrp_select(
    embeddings: np.ndarray,
    labeled_indices: set,
    budget: int,
    rng: np.random.Generator,
) -> list[int]:
    """
    One round of TPCRP selection (Algorithm 1, TPCRP variant).

    Args:
        embeddings      : (N, D) L2-normalised SimCLR embeddings
        labeled_indices : set of indices already in the labeled set
        budget          : B — number of points to select this round
        rng             : numpy RNG for reproducible K-means initialisation

    Returns:
        selected : list of B newly selected indices
    """
    N        = len(embeddings)
    n_labeled = len(labeled_indices)

    # Number of clusters: |L_{i-1}| + B, capped at MAX_CLUSTERS (Appendix F.1)
    n_clusters = min(n_labeled + budget, MAX_CLUSTERS)
    n_clusters = max(n_clusters, budget)   # always at least B clusters

    km_seed = int(rng.integers(0, 2**31))
    if n_clusters <= 50:
        km = KMeans(n_clusters=n_clusters, n_init=10, random_state=km_seed)
    else:
        km = MiniBatchKMeans(n_clusters=n_clusters, n_init=3, random_state=km_seed)

    cluster_ids = km.fit_predict(embeddings)  # (N,)

    # ── Find uncovered clusters ───────────────────────────────────────────────
    if labeled_indices:
        covered = set(cluster_ids[list(labeled_indices)])
    else:
        covered = set()
    uncovered = [c for c in range(n_clusters) if c not in covered]

    # Drop clusters with fewer than 5 samples (Appendix F.1 note)
    cluster_sizes = np.bincount(cluster_ids, minlength=n_clusters)
    uncovered = [c for c in uncovered if cluster_sizes[c] >= 5]

    if len(uncovered) == 0:
        # Fallback: pick uncovered regardless of size
        uncovered = [c for c in range(n_clusters) if c not in covered]

    # Sort uncovered clusters descending by size → take B largest
    uncovered_sorted = sorted(uncovered, key=lambda c: -cluster_sizes[c])
    target_clusters  = uncovered_sorted[:budget]

    # ── Pre-compute global KNN (once per round) ───────────────────────────────
    k_eff = min(KNN_K, N - 1)
    nbrs = NearestNeighbors(n_neighbors=k_eff + 1, metric="euclidean", n_jobs=-1)
    nbrs.fit(embeddings)

    all_indices = np.arange(N)
    selected    = []

    for c in target_clusters:
        cluster_mask          = cluster_ids == c
        cluster_point_indices = all_indices[cluster_mask].tolist()
        # Remove already-labeled points from consideration
        candidates = [idx for idx in cluster_point_indices if idx not in labeled_indices]
        if not candidates:
            continue

        # Typicality = 1 / mean dist to K (or cluster_size, whichever smaller) NNs
        # Use min(K, cluster_size) as per Appendix F.1
        k_use = min(KNN_K, len(candidates))
        query = embeddings[candidates]
        dists, _ = nbrs.kneighbors(query, n_neighbors=k_use + 1)
        mean_dists = dists[:, 1:k_use + 1].mean(axis=1)
        typicality = 1.0 / (mean_dists + 1e-10)

        best_idx = int(np.argmax(typicality))
        selected.append(candidates[best_idx])

    return selected


def random_select(
    labeled_indices: set,
    budget: int,
    n_total: int,
    rng: np.random.Generator,
) -> list[int]:
    """Select B points uniformly at random from the unlabelled pool."""
    unlabeled = np.array([i for i in range(n_total) if i not in labeled_indices])
    chosen    = rng.choice(unlabeled, size=min(budget, len(unlabeled)), replace=False)
    return chosen.tolist()


print("TPCRP and Random selection functions defined.")


## Cell 7 — ResNet-18 Classifier Training and Evaluation (Step 3)

For each labeled set, trains a **fresh** ResNet-18 (no pretrained weights) for 200 epochs.  
Optimiser: SGD with momentum 0.9, weight decay 5 * 10⁻⁴, initial lr 0.1, cosine annealing.  
Augmentations: random crop (padding=4), random horizontal flip, normalise.

In [ ]:
def get_clf_transforms():
    train_tf = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])
    test_tf = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])
    return train_tf, test_tf


def train_classifier(
    labeled_indices: set,
    seed: int,
) -> float:
    """
    Train a ResNet-18 on the given labeled subset and return test accuracy (%).

    The model is re-initialised from scratch for every call (paper Appendix F.2.1:
    'Re-Initialise weights between iterations').

    Args:
        labeled_indices : set of training-set indices to use as labeled data
        seed            : random seed for reproducibility

    Returns:
        test_acc : float — accuracy on the 10,000 CIFAR-10 test images (%)
    """
    set_seed(seed)

    train_tf, test_tf = get_clf_transforms()

    # ── Labeled training set ──────────────────────────────────────────────────
    full_train = torchvision.datasets.CIFAR10(
        root=DATA_DIR, train=True, download=True, transform=train_tf
    )
    labeled_ds    = Subset(full_train, sorted(labeled_indices))
    n_labeled     = len(labeled_indices)
    # Use a batch size of min(64, n_labeled) so batch never exceeds dataset size
    batch_size    = min(64, n_labeled)
    train_loader  = DataLoader(
        labeled_ds, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=False,
    )

    # ── Test set ──────────────────────────────────────────────────────────────
    test_ds = torchvision.datasets.CIFAR10(
        root=DATA_DIR, train=False, download=True, transform=test_tf
    )
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                             num_workers=2, pin_memory=True)

    # ── Model ─────────────────────────────────────────────────────────────────
    model     = resnet18(weights=None, num_classes=10).to(DEVICE)
    optimiser = optim.SGD(
        model.parameters(),
        lr=CLF_LR, momentum=CLF_MOMENTUM, weight_decay=CLF_WD, nesterov=True,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=CLF_EPOCHS)
    criterion = nn.CrossEntropyLoss()

    # ── Training loop ─────────────────────────────────────────────────────────
    model.train()
    for _ in range(CLF_EPOCHS):
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimiser.zero_grad()
            criterion(model(x), y).backward()
            optimiser.step()
        scheduler.step()

    # ── Evaluation ────────────────────────────────────────────────────────────
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y  = x.to(DEVICE), y.to(DEVICE)
            preds  = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total   += y.size(0)

    return 100.0 * correct / total


print("Classifier training function defined.")


## Cell 8 — Full AL Experiment: TPCRP vs Random Baseline (Step 2 + Step 3)

Runs **10 independent repetitions** of the complete active-learning loop.  
Each repetition:  
1. Starts with L₀ = ∅  
2. Runs 5 AL iterations (B = 10 per iteration)  
3. After each iteration trains a fresh ResNet-18 and records test accuracy  

The same loop is run for the **Random baseline** (uniform random selection).  
Results are cached to disk so the cell can be re-run without recomputing.

In [ ]:
results_path = os.path.join(OUTPUT_DIR, "al_results.pkl")

if os.path.exists(results_path):
    print("[Experiment] Loading cached results from disk …")
    with open(results_path, "rb") as fh:
        _saved = pickle.load(fh)
    tpcrp_accs  = _saved["tpcrp"]
    random_accs = _saved["random"]
    print(f"  tpcrp_accs  shape : {tpcrp_accs.shape}")
    print(f"  random_accs shape : {random_accs.shape}")
else:
    tpcrp_accs  = np.zeros((N_REPETITIONS, AL_ITERATIONS))
    random_accs = np.zeros((N_REPETITIONS, AL_ITERATIONS))

    print("=" * 60)
    print("Step 2 + 3 — TPCRP and Random AL Experiments")
    print(f"  Repetitions : {N_REPETITIONS}")
    print(f"  Iterations  : {AL_ITERATIONS}   Budget B = {AL_BUDGET}")
    print("=" * 60)

    for rep in range(N_REPETITIONS):
        rep_seed = 1000 + rep * 13
        rng      = np.random.default_rng(rep_seed)

        print(f"─── Repetition {rep + 1}/{N_REPETITIONS}  (seed={rep_seed}) ───")

        # ── TPCRP ─────────────────────────────────────────────────────────────
        labeled_set_tp = set()
        for it in range(AL_ITERATIONS):
            new_pts = tpcrp_select(embeddings, labeled_set_tp, AL_BUDGET, rng)
            labeled_set_tp.update(new_pts)
            clf_seed = rep_seed * 100 + it
            acc      = train_classifier(labeled_set_tp, seed=clf_seed)
            tpcrp_accs[rep, it] = acc
            label_dist = np.bincount(train_labels[sorted(labeled_set_tp)],
                                     minlength=10)
            print(f"  [TPCRP ] iter {it+1}  |L|={len(labeled_set_tp):3d}"
                  f"  dist={label_dist.tolist()}  acc={acc:.2f}%")

        # ── Random baseline ───────────────────────────────────────────────────
        labeled_set_rand = set()
        for it in range(AL_ITERATIONS):
            new_pts = random_select(labeled_set_rand, AL_BUDGET, N_TRAIN, rng)
            labeled_set_rand.update(new_pts)
            clf_seed = rep_seed * 100 + 50 + it
            acc      = train_classifier(labeled_set_rand, seed=clf_seed)
            random_accs[rep, it] = acc
            print(f"  [Random] iter {it+1}  |L|={len(labeled_set_rand):3d}"
                  f"  acc={acc:.2f}%")

    # ── Save ──────────────────────────────────────────────────────────────────
    with open(results_path, "wb") as fh:
        pickle.dump({"tpcrp": tpcrp_accs, "random": random_accs}, fh)
    print(f"[Experiment] Results saved → {results_path}")

# ── Summary statistics ────────────────────────────────────────────────────────
tpcrp_mean  = tpcrp_accs.mean(axis=0)
tpcrp_se    = tpcrp_accs.std(axis=0, ddof=1) / np.sqrt(N_REPETITIONS)
random_mean = random_accs.mean(axis=0)
random_se   = random_accs.std(axis=0, ddof=1) / np.sqrt(N_REPETITIONS)

print("" + "=" * 66)
print(f"{'Iter':>4} | {'|L|':>4} | {'TPCRP Mean':>10} | {'TPCRP SE':>8} "
      f"| {'Rand Mean':>9} | {'Rand SE':>7}")
print("-" * 66)
for i in range(AL_ITERATIONS):
    print(f"  {i+1:1d}  |  {(i+1)*AL_BUDGET:3d} | "
          f"  {tpcrp_mean[i]:7.2f}%  |  {tpcrp_se[i]:5.3f}%  | "
          f"  {random_mean[i]:6.2f}%  |  {random_se[i]:5.3f}%")
print("=" * 66)


## Cell 9 — Save Results Table (CSV)

In [ ]:
csv_path = os.path.join(OUTPUT_DIR, "results.csv")

with open(csv_path, "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow([
        "iteration", "n_labeled",
        "TPCRP_mean_acc", "TPCRP_std_err",
        "random_mean_acc", "random_std_err",
    ])
    for i in range(AL_ITERATIONS):
        writer.writerow([
            i + 1,
            (i + 1) * AL_BUDGET,
            round(float(tpcrp_mean[i]),  4),
            round(float(tpcrp_se[i]),    4),
            round(float(random_mean[i]), 4),
            round(float(random_se[i]),   4),
        ])

print(f"Results table saved → {csv_path}")

# Print for inspection
import csv as _csv
with open(csv_path) as fh:
    for row in _csv.reader(fh):
        print("  ", row)


## Cell 10 — Results Plot (Step 4)

Reproduces the style of Fig. 4(a) from the paper: mean test accuracy ± standard error vs AL iteration for both TPCRP and the Random baseline.

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go

iterations     = list(range(1, AL_ITERATIONS + 1))
n_labeled_list = [i * AL_BUDGET for i in iterations]
x_labels       = [f"Iter {i}<br>({i*AL_BUDGET} labels)" for i in iterations]

fig = go.Figure()

# ── TPCRP trace ───────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=iterations, y=tpcrp_mean.tolist(),
    mode="lines+markers",
    name="TPCRP (SimCLR + K-means)",
    line=dict(width=2.5),
    marker=dict(size=8, symbol="circle"),
    error_y=dict(type="data", array=tpcrp_se.tolist(), visible=True, thickness=1.5, width=6),
))

# ── TPCRP shaded SE band ─────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=iterations + iterations[::-1],
    y=(tpcrp_mean + tpcrp_se).tolist() + (tpcrp_mean - tpcrp_se).tolist()[::-1],
    fill="toself",
    fillcolor="rgba(99,110,250,0.15)",
    line=dict(color="rgba(255,255,255,0)"),
    showlegend=False,
    hoverinfo="skip",
))

# ── Random baseline trace ─────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=iterations, y=random_mean.tolist(),
    mode="lines+markers",
    name="Random",
    line=dict(width=2.5, dash="dash"),
    marker=dict(size=8, symbol="square"),
    error_y=dict(type="data", array=random_se.tolist(), visible=True, thickness=1.5, width=6),
))

# ── Random shaded SE band ─────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=iterations + iterations[::-1],
    y=(random_mean + random_se).tolist() + (random_mean - random_se).tolist()[::-1],
    fill="toself",
    fillcolor="rgba(239,85,59,0.15)",
    line=dict(color="rgba(255,255,255,0)"),
    showlegend=False,
    hoverinfo="skip",
))

fig.update_layout(
    title=dict(
        text=(
            "TPCRP vs Random — CIFAR-10 Low Budget"
            "<br><span style='font-size:15px;font-weight:normal;'>"
            "Source: Hacohen et al. ICML 2022 | Fully supervised, B=10, 10 reps</span>"
        ),
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
    xaxis=dict(
        tickmode="array",
        tickvals=iterations,
        ticktext=x_labels,
        title_text="AL Iteration",
    ),
    yaxis_title="Test Accuracy (%)",
)

fig.update_traces(cliponaxis=False)

plot_path = os.path.join(OUTPUT_DIR, "tpcrp_vs_random_cifar10.png")
fig.write_image(plot_path, scale=2)

import json
with open(plot_path + ".meta.json", "w") as fh:
    json.dump({
        "caption": "TPCRP vs Random: mean test accuracy ± SE across 10 repetitions on CIFAR-10",
        "description": (
            "Line chart comparing TPCRP (SimCLR + K-means) and random selection "
            "across 5 active learning iterations with B=10 labels per round. "
            "Shaded bands show ±1 standard error. Fully supervised ResNet-18 classifier."
        ),
    }, fh)

print(f"Plot saved → {plot_path}")
fig.show()
